In [159]:
from collections import defaultdict, namedtuple
import numpy as np
import pandas as pd
import math

In [160]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [161]:
import duolingo_replica_alex as dr
import duolingo as d

## Modelo 1: Réplica

Cargamos los datos y seleccionamos un subset de 5000 observaciones para testear el modelo.

In [131]:
data   = pd.read_csv('learning_traces.13m_en_10.csv' )
subset = data.sample( n = 50000, random_state = 5 )

Realizamos el split train-test, y además, obtenemos la lista de variables predictoras. Las variables predictoras incluyen: 'right', 'wrong', 'bias' y dicotómicas de lexemas.

In [132]:
trainset, testset, feature_vars = dr.read_data( subset, omit_lexemes = False )

# Predicting Probability of Recall - NN

In [133]:
trainset.time_afternoon

417754    1
395543    0
398659    0
409642    0
159919    0
         ..
455106    0
384728    0
276580    0
455794    0
90722     0
Name: time_afternoon, Length: 45000, dtype: uint8

In [134]:
# feature_vars_

In [135]:
feature_vars_time = feature_vars + ['time_afternoon', 'time_evening', 'time_morning', 'time_night']
feature_vars_time

['right',
 'wrong',
 'bias',
 "en:'/'<apos>",
 "en:'s/'s<gen>",
 'en:<*sf>/actor<n><*numb>',
 'en:<*sf>/advantage<n><*numb>',
 'en:<*sf>/afternoon<n><*numb>',
 'en:<*sf>/age<n><*numb>',
 'en:<*sf>/airport<n><*numb>',
 'en:<*sf>/answer<n><*numb>',
 'en:<*sf>/area<n><*numb>',
 'en:<*sf>/argument<n><*numb>',
 'en:<*sf>/aspect<n><*numb>',
 'en:<*sf>/attack<n><*numb>',
 'en:<*sf>/authority<n><*numb>',
 'en:<*sf>/baby<n><*numb>',
 'en:<*sf>/benefit<n><*numb>',
 'en:<*sf>/bicycle<n><*numb>',
 'en:<*sf>/body<n><*numb>',
 'en:<*sf>/bottle<n><*numb>',
 'en:<*sf>/box<n><*numb>',
 'en:<*sf>/brother<n><*numb>',
 'en:<*sf>/budget<n><*numb>',
 'en:<*sf>/building<n><*numb>',
 'en:<*sf>/cabinet<n><*numb>',
 'en:<*sf>/calendar<n><*numb>',
 'en:<*sf>/candidate<n><*numb>',
 'en:<*sf>/car<n><*numb>',
 'en:<*sf>/case<n><*numb>',
 'en:<*sf>/castle<n><*numb>',
 'en:<*sf>/cell<n><*numb>',
 'en:<*sf>/century<n><*numb>',
 'en:<*sf>/character<n><*numb>',
 'en:<*sf>/church<n><*numb>',
 'en:<*sf>/citizen<n><*numb>'

# Basic Model 

In [136]:
# Train dataset
y_train = trainset["p"]
X_train = trainset[feature_vars]
X_train["delta"] = trainset["t"]

# Test dataset
y_test = testset["p"]
X_test = testset[feature_vars]
X_test["delta"] = testset["t"]

C:\Users\ALEXAN~1\AppData\Local\Temp/ipykernel_47244/1339792806.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["delta"] = trainset["t"]
C:\Users\ALEXAN~1\AppData\Local\Temp/ipykernel_47244/1339792806.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["delta"] = testset["t"]


In [137]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Assuming you have preprocessed your data into X (features) and y (target)

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Adjust input_shape based on number of features
    Dense(1, activation='linear')
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Fit the model
model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.1)  # Adjust these parameters as needed

loss, mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {mae}")

Epoch 1/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0901 - mae: 0.1916 - val_loss: 0.0722 - val_mae: 0.1871
Epoch 2/200
1266/1266 [==============================] - 1s 944us/step - loss: 0.0743 - mae: 0.1706 - val_loss: 0.0807 - val_mae: 0.2058
Epoch 3/200
1266/1266 [==============================] - 1s 928us/step - loss: 0.0702 - mae: 0.1662 - val_loss: 0.0771 - val_mae: 0.1757
Epoch 4/200
1266/1266 [==============================] - 1s 930us/step - loss: 0.0698 - mae: 0.1654 - val_loss: 0.0727 - val_mae: 0.1478
Epoch 5/200
1266/1266 [==============================] - 1s 918us/step - loss: 0.0691 - mae: 0.1645 - val_loss: 0.0722 - val_mae: 0.1762
Epoch 6/200
1266/1266 [==============================] - 1s 906us/step - loss: 0.0675 - mae: 0.1624 - val_loss: 0.0728 - val_mae: 0.1745
Epoch 7/200
1266/1266 [==============================] - 1s 933us/step - loss: 0.0676 - mae: 0.1624 - val_loss: 0.0900 - val_mae: 0.2047
Epoch 8/200
1266/1266 [====================

Epoch 61/200
1266/1266 [==============================] - 1s 976us/step - loss: 0.0589 - mae: 0.1484 - val_loss: 0.0822 - val_mae: 0.1849
Epoch 62/200
1266/1266 [==============================] - 1s 973us/step - loss: 0.0585 - mae: 0.1472 - val_loss: 0.0827 - val_mae: 0.1814
Epoch 63/200
1266/1266 [==============================] - 1s 928us/step - loss: 0.0584 - mae: 0.1475 - val_loss: 0.0833 - val_mae: 0.1827
Epoch 64/200
1266/1266 [==============================] - 1s 971us/step - loss: 0.0582 - mae: 0.1470 - val_loss: 0.0835 - val_mae: 0.1756
Epoch 65/200
1266/1266 [==============================] - 1s 994us/step - loss: 0.0583 - mae: 0.1468 - val_loss: 0.0832 - val_mae: 0.1715
Epoch 66/200
1266/1266 [==============================] - 1s 946us/step - loss: 0.0584 - mae: 0.1473 - val_loss: 0.0838 - val_mae: 0.1824
Epoch 67/200
1266/1266 [==============================] - 1s 969us/step - loss: 0.0582 - mae: 0.1469 - val_loss: 0.0826 - val_mae: 0.1751
Epoch 68/200
1266/1266 [==========

1266/1266 [==============================] - 1s 906us/step - loss: 0.0568 - mae: 0.1447 - val_loss: 0.0922 - val_mae: 0.1834
Epoch 121/200
1266/1266 [==============================] - 1s 954us/step - loss: 0.0567 - mae: 0.1445 - val_loss: 0.0919 - val_mae: 0.1781
Epoch 122/200
1266/1266 [==============================] - 1s 927us/step - loss: 0.0568 - mae: 0.1440 - val_loss: 0.0924 - val_mae: 0.1897
Epoch 123/200
1266/1266 [==============================] - 1s 904us/step - loss: 0.0566 - mae: 0.1446 - val_loss: 0.0923 - val_mae: 0.1870
Epoch 124/200
1266/1266 [==============================] - 1s 967us/step - loss: 0.0566 - mae: 0.1445 - val_loss: 0.0919 - val_mae: 0.1774
Epoch 125/200
1266/1266 [==============================] - 1s 929us/step - loss: 0.0565 - mae: 0.1440 - val_loss: 0.0925 - val_mae: 0.1851
Epoch 126/200
1266/1266 [==============================] - 1s 935us/step - loss: 0.0566 - mae: 0.1443 - val_loss: 0.0912 - val_mae: 0.1760
Epoch 127/200
1266/1266 [================

1266/1266 [==============================] - 1s 967us/step - loss: 0.0557 - mae: 0.1429 - val_loss: 0.1026 - val_mae: 0.1860
Epoch 180/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0557 - mae: 0.1426 - val_loss: 0.1048 - val_mae: 0.2017
Epoch 181/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0557 - mae: 0.1425 - val_loss: 0.1038 - val_mae: 0.1946
Epoch 182/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0556 - mae: 0.1427 - val_loss: 0.1040 - val_mae: 0.1913
Epoch 183/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0558 - mae: 0.1435 - val_loss: 0.1056 - val_mae: 0.1883
Epoch 184/200
1266/1266 [==============================] - 2s 1ms/step - loss: 0.0556 - mae: 0.1425 - val_loss: 0.1019 - val_mae: 0.1977
Epoch 185/200
1266/1266 [==============================] - 1s 1ms/step - loss: 0.0555 - mae: 0.1426 - val_loss: 0.1050 - val_mae: 0.1863
Epoch 186/200
1266/1266 [============================

In [111]:
X_train

,right,wrong,bias,en:'/'<apos>,en:'s/'s<gen>,en:<*sf>/actor<n><*numb>,en:<*sf>/age<n><*numb>,en:<*sf>/area<n><*numb>,en:<*sf>/authority<n><*numb>,en:<*sf>/baby<n><*numb>,...,en:you/prpers<@ij:thank_you>,en:you/prpers<prn><obj><p2><mf><sp>,en:you/prpers<prn><subj><p2><mf><sp>,en:your/your<det><pos><sp>,en:yours/yours<prn><pos><mf><sp>,time_afternoon,time_evening,time_morning,time_night,delta
417754,2.000000,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0,0,1.199734
395543,3.316625,1.414214,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,1,0,0,3.568646
398659,1.732051,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,1,0.006921
409642,1.732051,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,1,0,2.428704
159919,3.316625,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,1,0,28.582280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141622,2.645751,1.732051,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0.155590
401929,2.828427,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,1,3.862095
179279,1.732051,1.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0.002442
424684,3.464102,1.732051,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,1,0,2.941076


# Adding Time-Section Features


In [108]:
# Train dataset
y_train = trainset["p"]
X_train = trainset[feature_vars_time]
X_train["delta"] = trainset["t"]

# Test dataset
y_test = testset["p"]
X_test = testset[feature_vars_time]
X_test["delta"] = testset["t"]

C:\Users\ALEXAN~1\AppData\Local\Temp/ipykernel_47244/1580624256.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["delta"] = trainset["t"]
C:\Users\ALEXAN~1\AppData\Local\Temp/ipykernel_47244/1580624256.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["delta"] = testset["t"]


In [110]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Assuming you have preprocessed your data into X (features) and y (target)

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Adjust input_shape based on number of features
    Dense(1, activation='linear')
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Fit the model
model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.1)  # Adjust these parameters as needed

loss, mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {mae}")

Epoch 1/200
127/127 [==============================] - 0s 2ms/step - loss: 0.2122 - mae: 0.3132 - val_loss: 0.0758 - val_mae: 0.1717
Epoch 2/200
127/127 [==============================] - 0s 896us/step - loss: 0.0692 - mae: 0.1626 - val_loss: 0.0717 - val_mae: 0.1823
Epoch 3/200
127/127 [==============================] - 0s 931us/step - loss: 0.0645 - mae: 0.1575 - val_loss: 0.0722 - val_mae: 0.1712
Epoch 4/200
127/127 [==============================] - 0s 951us/step - loss: 0.0614 - mae: 0.1510 - val_loss: 0.0733 - val_mae: 0.1771
Epoch 5/200
127/127 [==============================] - 0s 910us/step - loss: 0.0616 - mae: 0.1526 - val_loss: 0.0738 - val_mae: 0.1723
Epoch 6/200
127/127 [==============================] - 0s 918us/step - loss: 0.0606 - mae: 0.1507 - val_loss: 0.0800 - val_mae: 0.1920
Epoch 7/200
127/127 [==============================] - 0s 880us/step - loss: 0.0588 - mae: 0.1471 - val_loss: 0.1047 - val_mae: 0.2304
Epoch 8/200
127/127 [==============================] - 0s

127/127 [==============================] - 0s 916us/step - loss: 0.0318 - mae: 0.1042 - val_loss: 0.1094 - val_mae: 0.2204
Epoch 62/200
127/127 [==============================] - 0s 965us/step - loss: 0.0321 - mae: 0.1045 - val_loss: 0.1089 - val_mae: 0.2295
Epoch 63/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0314 - mae: 0.1038 - val_loss: 0.1092 - val_mae: 0.2271
Epoch 64/200
127/127 [==============================] - 0s 965us/step - loss: 0.0308 - mae: 0.1031 - val_loss: 0.1096 - val_mae: 0.2273
Epoch 65/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0319 - mae: 0.1064 - val_loss: 0.1087 - val_mae: 0.2159
Epoch 66/200
127/127 [==============================] - 0s 867us/step - loss: 0.0363 - mae: 0.1123 - val_loss: 0.1328 - val_mae: 0.2382
Epoch 67/200
127/127 [==============================] - 0s 914us/step - loss: 0.0357 - mae: 0.1158 - val_loss: 0.1136 - val_mae: 0.2363
Epoch 68/200
127/127 [==============================] - 0s 906us/

127/127 [==============================] - 0s 916us/step - loss: 0.0268 - mae: 0.0941 - val_loss: 0.1291 - val_mae: 0.2462
Epoch 122/200
127/127 [==============================] - 0s 880us/step - loss: 0.0273 - mae: 0.0980 - val_loss: 0.1250 - val_mae: 0.2502
Epoch 123/200
127/127 [==============================] - 0s 877us/step - loss: 0.0267 - mae: 0.0968 - val_loss: 0.1191 - val_mae: 0.2302
Epoch 124/200
127/127 [==============================] - 0s 893us/step - loss: 0.0275 - mae: 0.0982 - val_loss: 0.1227 - val_mae: 0.2368
Epoch 125/200
127/127 [==============================] - 0s 949us/step - loss: 0.0291 - mae: 0.1001 - val_loss: 0.1183 - val_mae: 0.2324
Epoch 126/200
127/127 [==============================] - 0s 954us/step - loss: 0.0258 - mae: 0.0928 - val_loss: 0.1221 - val_mae: 0.2342
Epoch 127/200
127/127 [==============================] - 0s 994us/step - loss: 0.0266 - mae: 0.0957 - val_loss: 0.1187 - val_mae: 0.2277
Epoch 128/200
127/127 [==============================] 

127/127 [==============================] - 0s 917us/step - loss: 0.0276 - mae: 0.0950 - val_loss: 0.1331 - val_mae: 0.2497
Epoch 182/200
127/127 [==============================] - 0s 917us/step - loss: 0.0238 - mae: 0.0901 - val_loss: 0.1256 - val_mae: 0.2407
Epoch 183/200
127/127 [==============================] - 0s 926us/step - loss: 0.0227 - mae: 0.0874 - val_loss: 0.1324 - val_mae: 0.2490
Epoch 184/200
127/127 [==============================] - 0s 922us/step - loss: 0.0236 - mae: 0.0904 - val_loss: 0.1309 - val_mae: 0.2489
Epoch 185/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0221 - mae: 0.0855 - val_loss: 0.1321 - val_mae: 0.2512
Epoch 186/200
127/127 [==============================] - 0s 972us/step - loss: 0.0225 - mae: 0.0867 - val_loss: 0.1362 - val_mae: 0.2593
Epoch 187/200
127/127 [==============================] - 0s 993us/step - loss: 0.0260 - mae: 0.0968 - val_loss: 0.1264 - val_mae: 0.2407
Epoch 188/200
127/127 [==============================] - 

In [112]:
X_train.columns

Index(['right', 'wrong', 'bias', 'en:'/'<apos>', 'en:'s/'s<gen>',
       'en:<*sf>/actor<n><*numb>', 'en:<*sf>/age<n><*numb>',
       'en:<*sf>/area<n><*numb>', 'en:<*sf>/authority<n><*numb>',
       'en:<*sf>/baby<n><*numb>',
       ...
       'en:you/prpers<@ij:thank_you>', 'en:you/prpers<prn><obj><p2><mf><sp>',
       'en:you/prpers<prn><subj><p2><mf><sp>', 'en:your/your<det><pos><sp>',
       'en:yours/yours<prn><pos><mf><sp>', 'time_afternoon', 'time_evening',
       'time_morning', 'time_night', 'delta'],
      dtype='object', length=795)

# Complex Model

In [113]:
from tensorflow.keras.layers import Dropout, BatchNormalization

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.1)

loss, mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {mae}")


Epoch 1/200
127/127 [==============================] - 0s 2ms/step - loss: 0.4453 - mae: 0.4845 - val_loss: 0.1642 - val_mae: 0.3808
Epoch 2/200
127/127 [==============================] - 0s 2ms/step - loss: 0.1803 - mae: 0.3163 - val_loss: 0.1082 - val_mae: 0.2794
Epoch 3/200
127/127 [==============================] - 0s 2ms/step - loss: 0.1401 - mae: 0.2727 - val_loss: 0.0867 - val_mae: 0.2481
Epoch 4/200
127/127 [==============================] - 0s 1ms/step - loss: 0.1140 - mae: 0.2443 - val_loss: 0.0764 - val_mae: 0.2212
Epoch 5/200
127/127 [==============================] - 0s 1ms/step - loss: 0.1062 - mae: 0.2306 - val_loss: 0.0736 - val_mae: 0.2059
Epoch 6/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0962 - mae: 0.2176 - val_loss: 0.0738 - val_mae: 0.1958
Epoch 7/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0918 - mae: 0.2096 - val_loss: 0.0718 - val_mae: 0.1935
Epoch 8/200
127/127 [==============================] - 0s 2ms/step - 

127/127 [==============================] - 0s 1ms/step - loss: 0.0469 - mae: 0.1249 - val_loss: 0.0900 - val_mae: 0.1631
Epoch 63/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0450 - mae: 0.1234 - val_loss: 0.0898 - val_mae: 0.1585
Epoch 64/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0458 - mae: 0.1235 - val_loss: 0.0881 - val_mae: 0.1672
Epoch 65/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0455 - mae: 0.1244 - val_loss: 0.0909 - val_mae: 0.1649
Epoch 66/200
127/127 [==============================] - 0s 2ms/step - loss: 0.0443 - mae: 0.1209 - val_loss: 0.0915 - val_mae: 0.1659
Epoch 67/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0433 - mae: 0.1205 - val_loss: 0.0960 - val_mae: 0.1625
Epoch 68/200
127/127 [==============================] - 0s 2ms/step - loss: 0.0431 - mae: 0.1194 - val_loss: 0.0963 - val_mae: 0.1879
Epoch 69/200
127/127 [==============================] - 0s 2ms/step - loss:

127/127 [==============================] - 0s 1ms/step - loss: 0.0338 - mae: 0.0986 - val_loss: 0.0983 - val_mae: 0.1711
Epoch 124/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0337 - mae: 0.0998 - val_loss: 0.1080 - val_mae: 0.1759
Epoch 125/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0326 - mae: 0.0976 - val_loss: 0.0984 - val_mae: 0.1662
Epoch 126/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0330 - mae: 0.0972 - val_loss: 0.1020 - val_mae: 0.1677
Epoch 127/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0320 - mae: 0.0948 - val_loss: 0.1030 - val_mae: 0.1724
Epoch 128/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0338 - mae: 0.0993 - val_loss: 0.0980 - val_mae: 0.1691
Epoch 129/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0351 - mae: 0.1018 - val_loss: 0.1004 - val_mae: 0.1609
Epoch 130/200
127/127 [==============================] - 0s 1ms/step 

Epoch 184/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0284 - mae: 0.0880 - val_loss: 0.0972 - val_mae: 0.1588
Epoch 185/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0292 - mae: 0.0891 - val_loss: 0.0999 - val_mae: 0.1610
Epoch 186/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0272 - mae: 0.0857 - val_loss: 0.1017 - val_mae: 0.1663
Epoch 187/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0299 - mae: 0.0914 - val_loss: 0.1030 - val_mae: 0.1748
Epoch 188/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0273 - mae: 0.0860 - val_loss: 0.1070 - val_mae: 0.1715
Epoch 189/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0280 - mae: 0.0876 - val_loss: 0.1002 - val_mae: 0.1567
Epoch 190/200
127/127 [==============================] - 0s 1ms/step - loss: 0.0275 - mae: 0.0853 - val_loss: 0.1056 - val_mae: 0.1855
Epoch 191/200
127/127 [==============================] 

# RNN

In [138]:
data   = pd.read_csv('learning_traces.13m_en_10.csv' )
subset = data.sample( n = 50000, random_state = 5 )

In [139]:
trainset, testset, feature_vars = dr.read_data( subset, omit_lexemes = False )

In [141]:
# Sort by user_id and timestamp/session identifier
trainset = trainset.sort_values(by=['user_id', 'timestamp'])

# Group by user_id
grouped = trainset.groupby('user_id')

sequences = []

for user_id, group in grouped:
    # Extract relevant features for each session
    # For example: ['feature1', 'feature2', ...]
    features = group[feature_vars_time].to_numpy()
    
    # Append the sequence of features for this user to the list
    sequences.append(features)

In [142]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad sequences
padded_sequences = pad_sequences(sequences, padding='post')

In [145]:
y.shape

(45000,)

In [149]:
padded_sequences

array([[[ 2,  1,  1, ...,  0,  0,  1],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0]],

       [[ 4,  2,  1, ...,  0,  0,  1],
        [15,  3,  1, ...,  0,  0,  1],
        [17,  4,  1, ...,  1,  0,  0],
        ...,
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0]],

       [[ 3,  1,  1, ...,  0,  0,  1],
        [27,  8,  1, ...,  0,  0,  1],
        [ 7,  2,  1, ...,  0,  0,  1],
        ...,
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0]],

       ...,

       [[16,  6,  1, ...,  1,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0

In [155]:
from sklearn.model_selection import train_test_split

# Assuming 'target' is your target variable
#y = trainset['p'].values  # You might need to adjust this to match your dataset's structure
y = trainset.groupby('user_id')['p'].mean().values  # Or any other aggregation that makes sense

X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=0.2, random_state=42)

In [156]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Define the RNN model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(32),
    Dense(1, activation='linear')  # or another activation, depending on your task
])

# Compile and fit the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
374/374 [==============================] - 249s 666ms/step - loss: 0.0606 - mae: 0.1600 - val_loss: 0.0453 - val_mae: 0.1312
Epoch 2/10
374/374 [==============================] - 240s 642ms/step - loss: 0.0432 - mae: 0.1402 - val_loss: 0.0455 - val_mae: 0.1589
Epoch 3/10
374/374 [==============================] - 244s 652ms/step - loss: 0.0433 - mae: 0.1403 - val_loss: 0.0454 - val_mae: 0.1308
Epoch 4/10
374/374 [==============================] - 246s 656ms/step - loss: 0.0435 - mae: 0.1406 - val_loss: 0.0447 - val_mae: 0.1414
Epoch 5/10
374/374 [==============================] - 250s 668ms/step - loss: 0.0432 - mae: 0.1400 - val_loss: 0.0449 - val_mae: 0.1509
Epoch 6/10
374/374 [==============================] - 252s 673ms/step - loss: 0.0431 - mae: 0.1401 - val_loss: 0.0455 - val_mae: 0.1589
Epoch 7/10
374/374 [==============================] - 256s 684ms/step - loss: 0.0429 - mae: 0.1394 - val_loss: 0.0446 - val_mae: 0.1439
Epoch 8/10
374/374 [============================

In [157]:
#loss, mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {mae}")

 31/104 [=======>......................] - ETA: 28s - loss: 0.0420 - mae: 0.1370

KeyboardInterrupt: 

In [150]:
print(f"Length of padded_sequences: {len(padded_sequences)}")
print(f"Length of y: {len(y)}")

Length of padded_sequences: 16611
Length of y: 45000


In [152]:
# Assuming 'p' is the column for target values
y = trainset.groupby('user_id')['p'].mean().values  # Or any other aggregation that makes sense
y.shape

(16611,)

In [154]:
import gc

# Delete large, unnecessary objects
del some_large_object

# Run garbage collection
gc.collect()


NameError: name 'some_large_object' is not defined